# Baseline v6

### 필요 라이브러리 설치

In [10]:
%pip install pandas numpy scikit-learn lightgbm xgboost

Note: you may need to restart the kernel to use updated packages.


### 데이터 불러오기 및 기본 전처리

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import joblib
import os
import lightgbm as lgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

# 폴더 세팅
os.makedirs('./model', exist_ok=True)
os.makedirs('./output', exist_ok=True)

print("1. 데이터 로딩 중...")
train = pd.read_csv('./data/train.csv')

print("2. 기초 결측치 처리 (0으로 통일)...")
numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if train[col].isnull().any():
        train[col] = train[col].fillna(0)

object_cols = train.select_dtypes(include=['object']).columns
for col in object_cols:
    if train[col].isnull().any():
        train[col] = train[col].fillna('Unknown')

print("3. 학습/검증 데이터 분리...")
X = train.drop(columns=['control_success', 'row_id'])
y = train['control_success']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ 완료! 학습용 크기: {X_train.shape}, 검증용 크기: {X_valid.shape}")

1. 데이터 로딩 중...
2. 기초 결측치 처리 (0으로 통일)...
3. 학습/검증 데이터 분리...


/var/folders/hm/v0t_41rn1k745bstq7qk0tn40000gn/T/ipykernel_66694/480720134.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = train.select_dtypes(include=['object']).columns


✅ 완료! 학습용 크기: (1180073, 47), 검증용 크기: (295019, 47)


In [12]:
print("1. 투수 Trackman 물리 데이터 및 구종 비율 장착 중...")
trackman = pd.read_csv('./data/trackman_history.csv', usecols=['pitcher_trackman_id', 'rel_speed', 'spin_rate', 'pitch_type_group'])

pitcher_physics = trackman.groupby('pitcher_trackman_id')[['rel_speed', 'spin_rate']].mean()
speed_dict = pitcher_physics['rel_speed'].to_dict()
spin_dict = pitcher_physics['spin_rate'].to_dict()
mean_speed = float(trackman['rel_speed'].mean())
mean_spin = float(trackman['spin_rate'].mean())

pitch_counts = trackman.groupby('pitcher_trackman_id').size()
top_pitch_counts = trackman.groupby('pitcher_trackman_id')['pitch_type_group'].apply(lambda x: x.value_counts().max())
primary_pitch_ratio = top_pitch_counts / pitch_counts
primary_pitch_dict = primary_pitch_ratio.to_dict()
mean_primary_ratio = float(primary_pitch_ratio.mean())

X_train['avg_speed'] = X_train['pitcher_id'].map(speed_dict).fillna(mean_speed)
X_train['avg_spin'] = X_train['pitcher_id'].map(spin_dict).fillna(mean_spin)
X_train['primary_pitch_ratio'] = X_train['pitcher_id'].map(primary_pitch_dict).fillna(mean_primary_ratio)

X_valid['avg_speed'] = X_valid['pitcher_id'].map(speed_dict).fillna(mean_speed)
X_valid['avg_spin'] = X_valid['pitcher_id'].map(spin_dict).fillna(mean_spin)
X_valid['primary_pitch_ratio'] = X_valid['pitcher_id'].map(primary_pitch_dict).fillna(mean_primary_ratio)

print("2. 타자 프로파일링(스무딩 타겟 인코딩) 장착 중...")
overall_mean = float(y_train.mean())
C = 10 # 스무딩 가중치

temp_train = X_train.copy()
temp_train['target'] = y_train
batter_stats = temp_train.groupby('batter_id')['target'].agg(['count', 'mean'])
batter_stats['batter_control_rate'] = (batter_stats['count'] * batter_stats['mean'] + C * overall_mean) / (batter_stats['count'] + C)

batter_dict = batter_stats['batter_control_rate'].to_dict()
X_train['batter_control_rate'] = X_train['batter_id'].map(batter_dict).fillna(overall_mean)
X_valid['batter_control_rate'] = X_valid['batter_id'].map(batter_dict).fillna(overall_mean)

print("3. 야구 상황 기반 파생 변수 생성 중...")
def add_baseball_features(df):
    df = df.copy() 
    df['pressure_index'] = 0
    df['pressure_index'] += np.where(df['inning'] >= 7, 1, 0)
    df['pressure_index'] += np.where(df['balls_before'] == 3, 2, 0)
    df['pressure_index'] += np.where((df['outs_before'] == 2) & (df['balls_before'] >= 2), 1, 0)
    df['count_advantage'] = df['strikes_before'] - df['balls_before']
    df['platoon_advantage'] = np.where(df['pitcher_hand'] == df['batter_hand'], 1, 0)
    return df

X_train = add_baseball_features(X_train)
X_valid = add_baseball_features(X_valid)

print("4. 모델 훈련용 파일 저장 중...")
joblib.dump(speed_dict, './model/speed_dict.pkl')
joblib.dump(spin_dict, './model/spin_dict.pkl')
joblib.dump(mean_speed, './model/mean_speed.pkl')
joblib.dump(mean_spin, './model/mean_spin.pkl')
joblib.dump(primary_pitch_dict, './model/primary_pitch_dict.pkl')
joblib.dump(mean_primary_ratio, './model/mean_primary_ratio.pkl')
joblib.dump(batter_dict, './model/batter_dict.pkl')
joblib.dump(overall_mean, './model/overall_mean.pkl')

print("5. 범주형 데이터 변환 중...")
object_cols = X_train.select_dtypes(include=['object']).columns
for col in object_cols:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')

print("6. LightGBM 모델 학습 시작...")
base_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
model.fit(X_train, y_train)

joblib.dump(model, './model/model.pkl')

print("\n7. 모의고사 채점 중...")
y_pred_proba = model.predict_proba(X_valid)[:, 1]
brier = brier_score_loss(y_valid, y_pred_proba)
bss = max(0, 100000 * (1 - brier / (overall_mean * (1 - overall_mean))))

print(f"🎯 V5(타자 스무딩 장착) 최종 평가 환산 점수: {bss:.2f}점")

1. 투수 Trackman 물리 데이터 및 구종 비율 장착 중...
2. 타자 프로파일링(스무딩 타겟 인코딩) 장착 중...
3. 야구 상황 기반 파생 변수 생성 중...
4. 모델 훈련용 파일 저장 중...
5. 범주형 데이터 변환 중...


/var/folders/hm/v0t_41rn1k745bstq7qk0tn40000gn/T/ipykernel_66694/3851514884.py:62: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = X_train.select_dtypes(include=['object']).columns


6. LightGBM 모델 학습 시작...
[LightGBM] [Info] Number of positive: 412201, number of negative: 374514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015912 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6658
[LightGBM] [Info] Number of data points in the train set: 786715, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.523952 -> initscore=0.095882
[LightGBM] [Info] Start training from score 0.095882
[LightGBM] [Info] Number of positive: 412201, number of negative: 374514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016392 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6655
[LightGBM] [Info] Number of data points in the train set: 786715, number of used fea

### 학습 모델 저장

In [13]:
script_content = """import pandas as pd
import numpy as np
import joblib
import os

def main():
    test = pd.read_csv('./data/test.csv')
    row_ids = test['row_id']
    X_test = test.drop(columns=['row_id'])
    
    # 1. 딕셔너리 및 평균값 로드
    speed_dict = joblib.load('./model/speed_dict.pkl')
    spin_dict = joblib.load('./model/spin_dict.pkl')
    mean_speed = joblib.load('./model/mean_speed.pkl')
    mean_spin = joblib.load('./model/mean_spin.pkl')
    primary_pitch_dict = joblib.load('./model/primary_pitch_dict.pkl')
    mean_primary_ratio = joblib.load('./model/mean_primary_ratio.pkl')
    batter_dict = joblib.load('./model/batter_dict.pkl')
    overall_mean = joblib.load('./model/overall_mean.pkl')
    
    # 2. 피처 매핑
    X_test['avg_speed'] = X_test['pitcher_id'].map(speed_dict).fillna(mean_speed)
    X_test['avg_spin'] = X_test['pitcher_id'].map(spin_dict).fillna(mean_spin)
    X_test['primary_pitch_ratio'] = X_test['pitcher_id'].map(primary_pitch_dict).fillna(mean_primary_ratio)
    X_test['batter_control_rate'] = X_test['batter_id'].map(batter_dict).fillna(overall_mean)
    
    # 3. 야구 도메인 변수 생성
    X_test['pressure_index'] = 0
    X_test['pressure_index'] += np.where(X_test['inning'] >= 7, 1, 0)
    X_test['pressure_index'] += np.where(X_test['balls_before'] == 3, 2, 0)
    X_test['pressure_index'] += np.where((X_test['outs_before'] == 2) & (X_test['balls_before'] >= 2), 1, 0)
    X_test['count_advantage'] = X_test['strikes_before'] - X_test['balls_before']
    X_test['platoon_advantage'] = np.where(X_test['pitcher_hand'] == X_test['batter_hand'], 1, 0)
    
    # 4. 결측치 및 범주형 처리
    numeric_cols = X_test.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if X_test[col].isnull().any():
            X_test[col] = X_test[col].fillna(0)
            
    object_cols = X_test.select_dtypes(include=['object']).columns
    for col in object_cols:
        if X_test[col].isnull().any():
            X_test[col] = X_test[col].fillna('Unknown')
        X_test[col] = X_test[col].astype('category')
        
    # 5. 모델 추론
    model = joblib.load('./model/model.pkl')
    pred_proba = model.predict_proba(X_test)[:, 1]
    
    os.makedirs('./output', exist_ok=True)
    pd.DataFrame({
        'row_id': row_ids,
        'control_success': pred_proba
    }).to_csv('./output/submission.csv', index=False)

if __name__ == '__main__':
    main()
"""

with open("script.py", "w", encoding="utf-8") as f:
    f.write(script_content)
    
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("lightgbm\nscikit-learn\n")

print("✅ V5 제출용 script.py 및 requirements.txt 파일 생성 완료!")

✅ V5 제출용 script.py 및 requirements.txt 파일 생성 완료!
